# Real-Time Customer Support Coach

Sits on top of a live customer conversation and does three things each turn:

1. **Analyze** the customer message -> sentiment, urgency, escalation risk, key issue, and the **order id / issue type** (the lookup keys).
2. **Look up facts** -> pull the customer's order record and the relevant policy from stub data sources.
3. **Suggest a grounded reply** -> The model drafts a reply using both conversation reasoning (tone/empathy) AND the looked-up facts (so it can state real numbers, not "I'll look into it").

When the agent replies, a fourth step **scores** their reply and gives one coaching tip.

The fact layer (`ORDERS`, `POLICIES`) is stubbed as plain dicts. Swap those two lookups for a real DB / API call later and nothing else changes.

> This is **not** RAG. It's keyed lookup: the analysis extracts an order id, we fetch that exact record. No semantic search, no embeddings.

> Logic lives in **`coach.py`**, imported below. `app.py` (the FastAPI backend) imports the same module, so the prompts and stub data exist in exactly one place.

In [ ]:
!pip install groq --quiet

# Colab note: this notebook imports from coach.py, which lives next to it in
# the repo. Upload coach.py alongside this notebook before running.

In [ ]:
import os
import json

# All coaching logic, prompts, and stub data live in coach.py — the single
# source of truth shared with app.py (the FastAPI backend). Nothing below is
# redefined here, so editing one copy can no longer drift from the other.
from coach import (
    AICoach,
    ConversationState,
    Message,
    CoachingFeedback,
    ORDERS,
    POLICIES,
    VALID_ISSUE_TYPES,
    get_order,
    get_policy,
    format_history,
)

## 1. Data models

Imported from `coach.py`. `ConversationState` stores `order_id` and
`issue_type` (extracted by the analyzer) so we know what to look up.

In [ ]:
# Defined in coach.py — shown here for reference.
import inspect
print(inspect.getsource(Message))
print(inspect.getsource(ConversationState))
print(inspect.getsource(CoachingFeedback))

## 2. Fact layer (STUBS)

Two keyed data sources, defined in `coach.py`. Today they're hand-written dicts. Later:
- `get_order()` -> a call to your order DB / helpdesk API
- `get_policy()` -> a call to your policy store

The function signatures stay the same, so the rest of the code never learns the difference.

**`issue_type` values the analyzer is allowed to use** are exactly the keys of `POLICIES`
(refund, shipping, damaged, technical, account). Constraining the model to this closed set
is what makes the lookup reliable.

In [ ]:
# ORDERS / POLICIES / get_order / get_policy all come from coach.py.
print("Orders on file:", list(ORDERS))
print("Valid issue types:", VALID_ISSUE_TYPES)
print()
print("Lookup check:", get_order("WORD-88221")["product"])
print("Policy check :", get_policy("refund"))

## 3. AI Coach

`AICoach` is defined in `coach.py` and imported above — the FastAPI backend uses
the exact same class and prompts.

Three responsibilities, and analyze + suggest are deliberately **one flow but two calls**:

The reply has to be grounded in facts we only know *after* analysis tells us the order id.
So the order is: analyze -> look up -> suggest. That's a data dependency, not a style
choice, so `suggest_reply` is a separate call that receives the looked-up facts.

`evaluate_agent_response` fires on a different event (the agent's turn), so it's independent.

Robustness: a retry wrapper around every API call, and text extraction that scans for the
text block instead of assuming `content[0]`.

In [ ]:
# AICoach lives in coach.py. Print it if you want to read the prompts here.
import inspect
print(inspect.getsource(AICoach))

## 4. Real-time session

One class (the original had two conflicting ones). It **returns** dicts so a frontend can consume them, and there's a small `print_*` helper for demo output so you still see something in the notebook.

Key behavior change from the original: a suggested reply is generated on **every** customer turn, not only when escalation risk is high. Coaching every reply was the whole point.

In [ ]:
class RealTimeCoachingSession:
    """Drives one conversation: customer turns get analysis + a grounded
    suggested reply; agent turns get scored."""

    def __init__(self):
        self.state = ConversationState()
        self.coach = AICoach()
        self.last_customer_message = ""

    def on_customer_message(self, message: str) -> dict:
        self.state.add_message("customer", message)
        self.last_customer_message = message

        analysis = self.coach.analyze_customer_message(message)

        self.state.sentiment = analysis["sentiment"]
        self.state.urgency = analysis["urgency"]
        self.state.escalation_risk = analysis["escalation_risk"]
        self.state.key_issue = analysis["key_issue"]
        self.state.order_id = analysis.get("order_id") or None
        self.state.issue_type = analysis.get("issue_type") or None

        # ---- fact lookups (keyed, not searched) ----
        order = get_order(self.state.order_id)
        policy = get_policy(self.state.issue_type)

        # ---- grounded suggested reply, every turn ----
        suggested_reply = self.coach.suggest_reply(
            customer_message=message,
            history_text=format_history(self.state.history),
            order=order,
            policy=policy,
        )

        return {
            "sentiment": self.state.sentiment,
            "urgency": self.state.urgency,
            "escalation_risk": self.state.escalation_risk,
            "key_issue": self.state.key_issue,
            "order_id": self.state.order_id,
            "issue_type": self.state.issue_type,
            "order_found": order is not None,
            "policy_found": policy is not None,
            "suggested_reply": suggested_reply,
        }

    def on_agent_message(self, message: str) -> dict:
        self.state.add_message("agent", message)
        feedback = self.coach.evaluate_agent_response(
            customer_message=self.last_customer_message,
            agent_message=message,
        )
        return {
            "tone_score": feedback.tone_score,
            "empathy_score": feedback.empathy_score,
            "clarity_score": feedback.clarity_score,
            "coaching_tip": feedback.coaching_tip,
        }


# ---- demo display helpers ----

def print_customer_result(msg: str, r: dict):
    print("\n" + "=" * 60)
    print("CUSTOMER:", msg)
    print("-" * 60)
    print(f"Sentiment: {r['sentiment']} | Urgency: {r['urgency']} | "
          f"Escalation: {r['escalation_risk']}")
    print(f"Key issue: {r['key_issue']}")
    print(f"Lookup keys -> order_id={r['order_id']} issue_type={r['issue_type']} "
          f"(order_found={r['order_found']}, policy_found={r['policy_found']})")
    print("\nSUGGESTED REPLY:")
    print(r["suggested_reply"])


def print_agent_result(msg: str, r: dict):
    print("\n" + "-" * 60)
    print("AGENT:", msg)
    print(f"Scores -> tone {r['tone_score']}/10  empathy {r['empathy_score']}/10  "
          f"clarity {r['clarity_score']}/10")
    print(f"Coaching tip: {r['coaching_tip']}")

## 5. Demo

`run_demo()` is now actually defined (the original called it but never defined it, which is what threw `NameError`).

The first customer message mentions order **WORD-88221**, so you should see `order_found=True, policy_found=True`, and the suggested reply should cite the real 20-day / 5-7 business day facts instead of a vague "I'll look into it".

In [ ]:
def run_demo():
    session = RealTimeCoachingSession()

    msg1 = ("I have been waiting for my refund on order WORD-88221 for 20 days! "
            "This is ridiculous. I want my money back immediately!")
    print_customer_result(msg1, session.on_customer_message(msg1))

    reply1 = "Please wait. We are checking your refund."
    print_agent_result(reply1, session.on_agent_message(reply1))

    msg2 = "How much longer do I have to wait?"
    print_customer_result(msg2, session.on_customer_message(msg2))

    reply2 = ("I completely understand your frustration. Waiting 20 days is longer "
              "than expected. I'll check the refund status and help with next steps.")
    print_agent_result(reply2, session.on_agent_message(reply2))


def run_interactive():
    print("\nREAL-TIME CUSTOMER SUPPORT COACH")
    print("Type 'customer: ...', 'agent: ...', or 'quit'.")
    session = RealTimeCoachingSession()
    while True:
        user_input = input("\n> ").strip()
        if user_input.lower() == "quit":
            print("Goodbye!")
            break
        if user_input.lower().startswith("customer:"):
            m = user_input[len("customer:"):].strip()
            if m:
                print_customer_result(m, session.on_customer_message(m))
        elif user_input.lower().startswith("agent:"):
            m = user_input[len("agent:"):].strip()
            if m:
                print_agent_result(m, session.on_agent_message(m))

In [ ]:
# Requires GROQ_API_KEY in the environment.
run_demo()

# For manual testing instead:
# run_interactive()